In [8]:
#importing 
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
import pandas as pd
import pickle
import numpy as np

In [4]:
# ── Load processed data ───────────────────────────────────────
syn_train = pd.read_csv("../data/processed/syn_train.csv")
syn_val   = pd.read_csv("../data/processed/syn_val.csv")
syn_test  = pd.read_csv("../data/processed/syn_test.csv")

In [5]:
# ── Load vocab ────────────────────────────────────────────────
with open("../models/deep_nn/vocab.pkl", "rb") as f:
    vocab = pickle.load(f)

# ── Load feature list ─────────────────────────────────────────
with open("../models/meta_features.pkl", "rb") as f:
    all_meta = pickle.load(f)

print(f"✅ Loaded | Vocab size: {len(vocab)}")

✅ Loaded | Vocab size: 10000


In [6]:
# ── Hyperparameters ───────────────────────────────────────────
MAX_LEN  = 100
VOCAB_SIZE = len(vocab)
EMBED_DIM  = 128

def encode_text(text, vocab, max_len):
    tokens  = text.lower().split()[:max_len]
    indices = [vocab.get(t, 1) for t in tokens]
    # pad
    indices += [0] * (max_len - len(indices))
    return indices


In [10]:
# ── Dataset ───────────────────────────────────────────────────
class ReviewDataset(Dataset):
    def __init__(self, df, vocab, max_len):
        self.texts  = [encode_text(t, vocab, max_len)
                       for t in df["review_text"]]
        self.meta   = df[all_meta].values.astype(np.float32)
        self.labels = df["label"].values

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "text":  torch.tensor(self.texts[idx], dtype=torch.long),
            "meta":  torch.tensor(self.meta[idx],  dtype=torch.float),
            "label": torch.tensor(self.labels[idx],dtype=torch.long),
        }

train_dataset = ReviewDataset(syn_train, vocab, MAX_LEN)
val_dataset   = ReviewDataset(syn_val,   vocab, MAX_LEN)
test_dataset  = ReviewDataset(syn_test,  vocab, MAX_LEN)

train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=32)
test_loader   = DataLoader(test_dataset,  batch_size=32)


In [11]:
#Defining the model
# ── Model 1: BiLSTM ───────────────────────────────────────────
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim,
                 meta_dim, num_classes=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim, batch_first=True,
                                 bidirectional=True, num_layers=2,
                                 dropout=dropout)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_dim * 2 + meta_dim, num_classes)

    def forward(self, text, meta):
        emb         = self.dropout(self.embedding(text))
        out, (h, _) = self.lstm(emb)
        # concat last hidden states of both directions
        h_cat       = torch.cat([h[-2], h[-1]], dim=1)
        combined    = torch.cat([h_cat, meta], dim=1)
        return self.fc(self.dropout(combined))

# ── Model 2: BiLSTM + Attention ───────────────────────────────
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn    = nn.Linear(hidden_dim * 2, 1)

    def forward(self, lstm_out):
        scores  = self.attn(lstm_out).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        context = (lstm_out * weights.unsqueeze(-1)).sum(dim=1)
        return context, weights

class BiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim,
                 meta_dim, num_classes=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim, batch_first=True,
                                 bidirectional=True, num_layers=2,
                                 dropout=dropout)
        self.attention = Attention(hidden_dim)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_dim * 2 + meta_dim, num_classes)

    def forward(self, text, meta):
        emb          = self.dropout(self.embedding(text))
        lstm_out, _  = self.lstm(emb)
        context, _   = self.attention(lstm_out)
        combined     = torch.cat([context, meta], dim=1)
        return self.fc(self.dropout(combined))

# ── Model 3: CNN + BiLSTM ─────────────────────────────────────
class CNNBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim,
                 meta_dim, num_classes=2, dropout=0.3):
        super().__init__()
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.conv       = nn.Conv1d(embed_dim, 128, kernel_size=3, padding=1)
        self.lstm       = nn.LSTM(128, hidden_dim, batch_first=True,
                                  bidirectional=True, dropout=dropout)
        self.dropout    = nn.Dropout(dropout)
        self.fc         = nn.Linear(hidden_dim * 2 + meta_dim, num_classes)

    def forward(self, text, meta):
        emb         = self.dropout(self.embedding(text))
        # CNN expects (batch, channels, seq_len)
        cnn_out     = torch.relu(self.conv(emb.permute(0, 2, 1)))
        cnn_out     = cnn_out.permute(0, 2, 1)
        _, (h, _)   = self.lstm(cnn_out)
        h_cat       = torch.cat([h[-2], h[-1]], dim=1)
        combined    = torch.cat([h_cat, meta], dim=1)
        return self.fc(self.dropout(combined))
